# Phase 5 Validation — Workflow Orchestrator

**Purpose:** Interactively demonstrate the LangGraph StateGraph executing the full
workflow with all agents mocked. No real API calls — every LLM response is
simulated so you can see routing, state transitions, HITL interrupts, budget
enforcement, and error isolation without any cost.

## What this notebook shows

| Section | Topic |
|---|---|
| 1 | Setup + shared mock helpers |
| 2 | WorkflowGraphState — structure and defaults |
| 3 | Execution limits — constants and budget helpers |
| 4 | Individual nodes — each node called in isolation |
| 5 | Conditional routers — interview and tailoring routing |
| 6 | Full graph run — happy path, all mocked agents |
| 7 | HITL simulation — pause at job selection, resume with decision |
| 8 | Error isolation — per-job LLMProviderError, run continues |
| 9 | Budget exhaustion — remaining jobs marked budget_skipped |
| 10 | PSSR checklist — assertions verifying Phase 5 invariants |
| 11 | End-to-end agent pipeline — all 8 agents in sequence with preconditions, context, and output verification |
| 12 | Graph visualizations — compiled StateGraph rendered by LangGraph |

---
## Section 1 — Setup

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
import os as _os; _os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from unittest.mock import MagicMock
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from app.providers.llm_client import LLMClient, LLMProviderError
from app.services.observability_service import ObservabilityService
from app.services.job_discovery_service import JobDiscoveryService
from app.services.resume_parser import ResumeParser
from app.services.report_generator import ReportGenerator
from app.repositories.job_repository import JobRepository
from app.repositories.score_repository import ScoreRepository
from app.repositories.advice_repository import AdviceRepository
from app.repositories.review_repository import ReviewRepository
from app.repositories.tailoring_repository import TailoringRepository
from app.repositories.workflow_repository import WorkflowRepository
from app.repositories.resume_repository import ResumeRepository

from app.agents.research_agent import ResearchAgent
from app.agents.scoring_agent import ScoringAgent
from app.agents.resume_critic import ResumeCritic
from app.agents.review_auditor import ReviewAuditor
from app.agents.career_advisor import CareerAdvisor
from app.agents.interview_coach import InterviewCoach
from app.agents.tailoring_agent import TailoringAgent
from app.agents.fidelity_reviewer import FidelityReviewer

from app.schemas.job_score import JobScore
from app.schemas.research_context import ResearchContext
from app.schemas.resume_review import ResumeReview
from app.schemas.review_audit import ReviewAudit
from app.schemas.career_advice import CareerAdvice
from app.schemas.interview_prep import InterviewPrep
from app.schemas.tailored_resume_draft import TailoredResumeDraft
from app.schemas.fidelity_review import FidelityReview
from app.schemas.job_posting import JobPosting, JobSource, WorkMode
from app.repositories.database import utcnow_iso

from app.workflows.workflow_graph import WorkflowDependencies, build_graph
from app.workflows.graph_state import WorkflowGraphState
from app.workflows.limits import (
    MAX_JOBS_PER_RUN, MIN_MATCH_SCORE_DEFAULT, MAX_SELECTED_JOBS, MAX_REVIEW_ROUNDS,
    MAX_LLM_CALLS_PER_RUN,
    AUDIT_QUALITY_THRESHOLD, STAGNATION_MIN_IMPROVEMENT,
)

print("All imports OK")

In [ ]:
# ── Shared mock helpers ──────────────────────────────────────────────────────

WF_ID = "wf-demo-001"

RESUME_PROFILE = {
    "name": "Jane Smith",
    "skills": ["Python", "Kubernetes", "GCP"],
    "experience": [{"title": "Senior Engineer", "company": "Payments Co", "years": 4}],
}

JOB_POSTING = JobPosting(
    job_id="job-001", workflow_id=WF_ID,
    url="https://example.com/job", source=JobSource.MANUAL,
    title="Staff Engineer", company="FinTech Corp",
    work_mode=WorkMode.REMOTE, description="Python, Kubernetes, distributed systems.",
    found_at=utcnow_iso(),
)

def make_obs():
    obs = MagicMock(spec=ObservabilityService)
    obs.log_agent_started.return_value = "evt-mock-001"
    return obs

def make_agent(agent_class, return_schema):
    """Return a mock agent whose run() returns the given Pydantic schema instance."""
    mock = MagicMock(spec=agent_class)
    mock.run.return_value = return_schema
    return mock

# Pre-built schema instances for use throughout
RESEARCH  = ResearchContext(job_id="job-001", company_summary="Tech co.",
    role_context="Platform.", technology_signals=["Python"], leadership_signals=[],
    domain_signals=[], risk_flags=[], research_steps=[], confidence=75)

SCORE = JobScore(job_id="job-001", resume_id="res-001",
    overall_score=82, technical_score=88, architecture_score=75,
    leadership_score=60, domain_score=70, match_summary="Strong technical fit.",
    strengths=["Python"], gaps=["Leadership scope"],
    recommended_next_action="Apply.", confidence=85)

REVIEW = ResumeReview(job_id="job-001", resume_id="res-001",
    overall_fit_summary="Good technical fit.", section_reviews=[],
    critical_gaps=["No management exp"], resume_only_gaps=["Scale data missing"],
    career_gaps_observed=["No direct reports"],
    suggested_improvements=["Quantify K8s migration"],
    questions_for_user=["How many teams did you coordinate?"], confidence=80)

AUDIT = ReviewAudit(job_id="job-001", round_number=1,
    audit_score=82, auditor_confidence=80, quality_summary="Sufficient quality.",
    missing_analysis_points=[], generic_or_weak_feedback=[],
    unsupported_claims=[], fidelity_concerns=[],
    recommended_revision_instructions=[], stop_recommendation=True,
    stop_reason="Quality threshold reached.")

ADVICE = CareerAdvice(job_id="job-001",
    positioning_summary="Lead with distributed systems depth.",
    resume_gaps=["Scale data missing from bullets"],
    career_gaps=["No direct reports — cannot be tailored"],
    role_fit_assessment="High fit for IC track.",
    recommended_positioning="Lead with platform depth.",
    skills_to_strengthen=["Staff-level design"], experience_to_collect=["Lead cross-team initiative"],
    thirty_sixty_ninety_day_plan=["30d: identify initiative"],
    recommended_next_action="Apply.", confidence=82)

PREP = InterviewPrep(job_id="job-001",
    likely_interview_topics=["Distributed system design"],
    technical_topics_to_review=["Raft consensus"], leadership_stories_to_prepare=[],
    weak_areas_to_defend=["No direct reports"],
    questions_to_ask_interviewer=["What does success look like in 90 days?"],
    seven_day_prep_plan=["Day 1-2: review distributed systems fundamentals"],
    confidence=85)

DRAFT = TailoredResumeDraft(job_id="job-001", resume_id="res-001",
    summary_suggestions=[], experience_bullet_suggestions=[],
    skills_section_suggestions=["Add: Distributed Systems"],
    overall_tailoring_notes="Strong rewords possible.",
    fidelity_risk_summary="Low risk.")

FIDELITY = FidelityReview(job_id="job-001", resume_id="res-001",
    overall_fidelity_status="pass", unsupported_claims=[],
    fabricated_metrics=[], inflated_scope_flags=[],
    unsupported_technology_flags=[], unsupported_certification_flags=[],
    required_removals=[], required_revisions=[],
    approval_recommendation="approve", confidence=95)

print("Mock helpers ready")
print(f"Candidate: {RESUME_PROFILE['name']}")
print(f"Job:       {JOB_POSTING.title} @ {JOB_POSTING.company}")

---
## Section 2 — WorkflowGraphState

In [ ]:
import inspect
from app.workflows.graph_state import WorkflowGraphState

print("WorkflowGraphState fields:")
hints = WorkflowGraphState.__annotations__
for group, keys in [
    ("Identity",          ["workflow_id", "workflow_type", "status", "current_step"]),
    ("Resume",            ["resume_id", "resume_profile", "resume_version"]),
    ("Jobs",              ["normalized_jobs", "scored_jobs", "selected_jobs"]),
    ("Review",            ["review_rounds", "final_resume_review"]),
    ("Career Intel",      ["career_advice", "interview_prep", "tailored_resume", "fidelity_review"]),
    ("HITL",              ["pending_decision", "human_decisions"]),
    ("Metrics",           ["run_metrics", "errors"]),
    ("Routing flags",     ["user_requested_interview_prep", "user_requested_tailoring"]),
]:
    print(f"\n  [{group}]")
    for k in keys:
        print(f"    {k}: {hints.get(k, '?')}")

print(f"\nTotal fields: {len(hints)}")
print("All fields optional (total=False) — nodes return partial updates.")

---
## Section 3 — Execution Limits

In [ ]:
from app.workflows.limits import (
    BudgetExceededError, check_budget, add_llm_call, get_metrics, append_error
)

print("Execution limits (CLAUDE.md invariants):")
print(f"  MAX_JOBS_PER_RUN        = {MAX_JOBS_PER_RUN}")
print(f"  MAX_SELECTED_JOBS       = {MAX_SELECTED_JOBS}")
print(f"  MAX_REVIEW_ROUNDS       = {MAX_REVIEW_ROUNDS}")
print(f"  MAX_LLM_CALLS_PER_RUN   = {MAX_LLM_CALLS_PER_RUN}")
print(f"  MIN_MATCH_SCORE_DEFAULT = {MIN_MATCH_SCORE_DEFAULT}")
print(f"  AUDIT_QUALITY_THRESHOLD = {AUDIT_QUALITY_THRESHOLD}")
print(f"  STAGNATION_MIN_IMPROVEMENT = {STAGNATION_MIN_IMPROVEMENT}")
print()

# Show budget check behaviour
ok_state    = {"run_metrics": {"llm_calls": 10}}
full_state  = {"run_metrics": {"llm_calls": MAX_LLM_CALLS_PER_RUN}}

check_budget(ok_state)   # should not raise
print("check_budget(llm_calls=10): OK")

try:
    check_budget(full_state)
except BudgetExceededError as e:
    print(f"check_budget(llm_calls={MAX_LLM_CALLS_PER_RUN}): BudgetExceededError — {e}")

# Show metric accumulation
metrics = {"llm_calls": 3, "tokens_input": 1000, "tokens_output": 200,
           "estimated_cost_usd": 0.002, "total_duration_ms": 0,
           "started_at": None, "completed_at": None}
updated = add_llm_call(metrics, tokens_in=500, tokens_out=100, cost_usd=0.001)
print(f"\nadd_llm_call: llm_calls {metrics['llm_calls']} → {updated['llm_calls']}")
print(f"             cost ${metrics['estimated_cost_usd']:.4f} → ${updated['estimated_cost_usd']:.4f}")

---
## Section 4 — Individual Nodes

In [ ]:
# ── discover_jobs ─────────────────────────────────────────────────────────────
from app.workflows.nodes.discover_jobs import make_discover_jobs_node

discovery_svc = MagicMock(spec=JobDiscoveryService)
discovery_svc.discover.return_value = [JOB_POSTING]

node = make_discover_jobs_node(discovery_svc, MagicMock(spec=JobRepository), make_obs())
result = node({"workflow_id": WF_ID, "search_criteria": {"roles": ["Staff Engineer"]},
               "errors": []})

print("discover_jobs:")
print(f"  normalized_jobs count : {len(result['normalized_jobs'])}")
print(f"  first job id          : {result['normalized_jobs'][0]['id']}")
print(f"  first job status      : {result['normalized_jobs'][0]['status']}")
print(f"  current_step          : {result['current_step']}")
assert result['normalized_jobs'][0]['status'] == 'discovered'
print("  → assertion passed")

In [ ]:
# ── score_jobs ────────────────────────────────────────────────────────────────
from app.workflows.nodes.score_jobs import make_score_jobs_node

research_mock = make_agent(ResearchAgent, RESEARCH)
scoring_mock  = make_agent(ScoringAgent,  SCORE)

job_dict = {
    "id": "job-001", "job_id": "job-001",
    "title": "Staff Engineer", "company": "FinTech Corp",
    "job_description": JOB_POSTING.description,
    "url": JOB_POSTING.url, "location": "Remote", "status": "discovered",
}

node = make_score_jobs_node(research_mock, scoring_mock, MagicMock(spec=ScoreRepository), make_obs())
result = node({
    "workflow_id": WF_ID, "resume_id": "res-001",
    "resume_profile": RESUME_PROFILE,
    "normalized_jobs": [job_dict],
    "run_metrics": {"llm_calls": 0, "tokens_input": 0, "tokens_output": 0,
                    "estimated_cost_usd": 0.0},
    "errors": [],
    "effective_config": {"scoring": {"career_track": "ic"}},
})

print("score_jobs:")
scored = result['scored_jobs'][0]
print(f"  job status      : {scored['status']}")
print(f"  overall_score   : {scored['overall_score']}/100")
print(f"  technical_score : {scored['technical_score']}/100")
print(f"  llm_calls used  : {result['run_metrics']['llm_calls']}  (1 research + 1 scoring)")
assert scored['status'] == 'scored'
assert result['run_metrics']['llm_calls'] == 2
print("  → assertions passed")

In [ ]:
# ── deep_review (reflection loop) ────────────────────────────────────────────
from app.workflows.nodes.deep_review import make_deep_review_node

scored_job = {**job_dict, "status": "scored", "overall_score": 82,
              "job_id": "job-001", "resume_id": "res-001"}

node = make_deep_review_node(
    make_agent(ResumeCritic,   REVIEW),
    make_agent(ReviewAuditor,  AUDIT),
    MagicMock(spec=ReviewRepository),
    make_obs(),
)
result = node({
    "workflow_id": WF_ID, "resume_id": "res-001",
    "resume_profile": RESUME_PROFILE,
    "selected_jobs": [scored_job],
    "scored_jobs": [scored_job],
    "run_metrics": {"llm_calls": 2, "tokens_input": 0, "tokens_output": 0, "estimated_cost_usd": 0.0},
    "errors": [],
})

print("deep_review:")
print(f"  review_rounds count    : {len(result['review_rounds'])}")
print(f"  final_review summary   : {result['final_resume_review']['overall_fit_summary']}")
print(f"  loop stopped because   : {result['review_rounds'][0]['stop_reason']}")
print(f"  current_step           : {result['current_step']}")
assert result['final_resume_review'] is not None
print("  → assertion passed")

In [ ]:
# ── career_advice, interview_prep, tailoring, generate_report ─────────────────
from app.workflows.nodes.career_advice  import make_career_advice_node
from app.workflows.nodes.interview_prep import make_interview_prep_node
from app.workflows.nodes.tailoring      import make_tailoring_node
from app.workflows.nodes.generate_report import make_generate_report_node

base = {
    "workflow_id": WF_ID, "resume_id": "res-001",
    "resume_profile": RESUME_PROFILE,
    "selected_jobs": [scored_job], "scored_jobs": [scored_job],
    "final_resume_review": REVIEW.model_dump(),
    "career_advice": ADVICE.model_dump(),
    "run_metrics": {"llm_calls": 4, "tokens_input": 0, "tokens_output": 0,
                    "estimated_cost_usd": 0.0},
    "errors": [],
    "effective_config": {"scoring": {"career_track": "ic"}},
    "user_requested_interview_prep": True,
    "user_requested_tailoring": True,
}

# CareerAdvisor
adv_result = make_career_advice_node(make_agent(CareerAdvisor, ADVICE),
                                      MagicMock(spec=AdviceRepository), make_obs())(base)
print(f"career_advice : positioning = {adv_result['career_advice']['positioning_summary']}")

# InterviewCoach
prep_result = make_interview_prep_node(make_agent(InterviewCoach, PREP),
                                        MagicMock(spec=AdviceRepository), make_obs())(base)
print(f"interview_prep: topics = {prep_result['interview_prep']['likely_interview_topics']}")

# TailoringAgent + FidelityReviewer
tail_result = make_tailoring_node(
    make_agent(TailoringAgent,    DRAFT),
    make_agent(FidelityReviewer,  FIDELITY),
    MagicMock(spec=TailoringRepository), make_obs(),
)(base)
print(f"tailoring     : fidelity_status = {tail_result['fidelity_review']['overall_fidelity_status']}")
print(f"              : approval = {tail_result['fidelity_review']['approval_recommendation']}")

# ReportGenerator
rg = MagicMock(spec=ReportGenerator)
rg.generate_run_summary.return_value = "# Run Report\n\nAll done."
rep_result = make_generate_report_node(rg, make_obs())(base)
print(f"generate_report: status={rep_result['status']}, report_len={len(rep_result['report']['markdown'])} chars")

print("\nAll node assertions passed")

---
## Section 5 — Conditional Routers

In [ ]:
from app.workflows.routers import interview_router, tailoring_router

print(f"interview_router thresholds: score >= {MIN_MATCH_SCORE_DEFAULT}")
print()

cases = [
    ({"scored_jobs": [{"overall_score": 85}], "user_requested_interview_prep": False}, interview_router, "interview_prep"),
    ({"scored_jobs": [{"overall_score": 40}], "user_requested_interview_prep": False}, interview_router, "tailoring_check"),
    ({"scored_jobs": [{"overall_score": 10}], "user_requested_interview_prep": True},  interview_router, "interview_prep"),
    ({"user_requested_tailoring": True},  tailoring_router, "tailoring"),
    ({"user_requested_tailoring": False}, tailoring_router, "generate_report"),
]

for state, router, expected in cases:
    result = router(state)
    status = "PASS" if result == expected else "FAIL"
    score_info = f"score={state.get('scored_jobs', [{}])[0].get('overall_score', 'n/a')}" \
                 if 'scored_jobs' in state else f"tailoring_requested={state.get('user_requested_tailoring')}"
    print(f"  [{status}] {router.__name__}({score_info}) → {result}")

print("\nAll router assertions passed")

---
## Section 6 — Full Graph Run (Happy Path)

In [ ]:
def make_deps(checkpointer=None, scoring_score: int = 82) -> WorkflowDependencies:
    """Build WorkflowDependencies with all agents and services mocked."""
    score_inst = JobScore(
        job_id="job-001", resume_id="res-001",
        overall_score=scoring_score, technical_score=88, architecture_score=75,
        leadership_score=60, domain_score=70, match_summary="Good.",
        strengths=["Python"], gaps=[], recommended_next_action="Apply.", confidence=85,
    )
    disc = MagicMock(spec=JobDiscoveryService)
    disc.discover.return_value = [JOB_POSTING]
    rg = MagicMock(spec=ReportGenerator)
    rg.generate_run_summary.return_value = "# Report"
    return WorkflowDependencies(
        research_agent   = make_agent(ResearchAgent,  RESEARCH),
        scoring_agent    = make_agent(ScoringAgent,   score_inst),
        resume_critic    = make_agent(ResumeCritic,   REVIEW),
        review_auditor   = make_agent(ReviewAuditor,  AUDIT),
        career_advisor   = make_agent(CareerAdvisor,  ADVICE),
        interview_coach  = make_agent(InterviewCoach, PREP),
        tailoring_agent  = make_agent(TailoringAgent, DRAFT),
        fidelity_reviewer= make_agent(FidelityReviewer, FIDELITY),
        discovery_service= disc,
        resume_parser    = MagicMock(spec=ResumeParser),
        report_generator = rg,
        job_repo         = MagicMock(spec=JobRepository),
        score_repo       = MagicMock(spec=ScoreRepository),
        advice_repo      = MagicMock(spec=AdviceRepository),
        review_repo      = MagicMock(spec=ReviewRepository),
        tailoring_repo   = MagicMock(spec=TailoringRepository),
        workflow_repo    = MagicMock(spec=WorkflowRepository),
        resume_repo      = MagicMock(spec=ResumeRepository),
        observability    = make_obs(),
        checkpointer     = checkpointer or MemorySaver(),
    )

def initial_state(wf_id: str, **overrides) -> dict:
    state = {
        "workflow_id": wf_id, "workflow_type": "full_career_review",
        "status": "running", "current_step": "initialized",
        "resume_id": "res-001", "resume_profile": RESUME_PROFILE,
        "search_criteria": {"roles": ["Staff Engineer"]},
        "normalized_jobs": [], "scored_jobs": [], "selected_jobs": [],
        "run_metrics": {"llm_calls": 0, "tokens_input": 0, "tokens_output": 0,
                        "estimated_cost_usd": 0.0},
        "errors": [], "effective_config": {"scoring": {"career_track": "ic"}},
        "human_decisions": [],
        "user_requested_interview_prep": False,
        "user_requested_tailoring": False,
        "created_at": utcnow_iso(), "updated_at": utcnow_iso(),
    }
    state.update(overrides)
    return state

print("Helpers built — make_deps() and initial_state() ready")

In [ ]:
# Build graph and run to first HITL interrupt (job selection)
saver = MemorySaver()
deps  = make_deps(checkpointer=saver)
graph = build_graph(deps)

config = {"configurable": {"thread_id": "wf-demo-happy"}}
state  = initial_state("wf-demo-happy")

print("Running graph (will pause at await_job_selection)...")
try:
    result = graph.invoke(state, config)
    print(f"Graph completed or paused. Current step: {result.get('current_step', '?')}")
except Exception as exc:
    print(f"Graph paused with: {type(exc).__name__}")

# Check scoring ran
deps.scoring_agent.run.assert_called()
deps.research_agent.run.assert_called()
print(f"ResearchAgent called : {deps.research_agent.run.call_count} time(s)")
print(f"ScoringAgent called  : {deps.scoring_agent.run.call_count} time(s)")

---
## Section 7 — HITL Simulation

In [ ]:
# Resume the workflow with job selection decision
print("Resuming with job selection: ['job-001']")
try:
    result = graph.invoke(
        Command(resume={"selected_job_ids": ["job-001"]}),
        config,
    )
    print(f"After resume — current_step: {result.get('current_step', '?')}")
    print(f"After resume — status: {result.get('status', '?')}")
except Exception as exc:
    print(f"Paused again with: {type(exc).__name__} (second HITL checkpoint)")

# Deep review should have run
deps.resume_critic.run.assert_called()
deps.review_auditor.run.assert_called()
deps.career_advisor.run.assert_called()
print(f"ResumeCritic called   : {deps.resume_critic.run.call_count} time(s)")
print(f"ReviewAuditor called  : {deps.review_auditor.run.call_count} time(s)")
print(f"CareerAdvisor called  : {deps.career_advisor.run.call_count} time(s)")
print()
print("HITL flow:")
print("  graph.invoke(initial_state) → paused at await_job_selection")
print("  graph.invoke(Command(resume={...})) → continued from checkpoint")

---
## Section 8 — Error Isolation

In [ ]:
# Scoring fails for one job — run should continue with that job marked as failed
from app.workflows.nodes.score_jobs import make_score_jobs_node

failing_scoring = MagicMock(spec=ScoringAgent)
failing_scoring.run.side_effect = LLMProviderError("API timeout")

jobs = [
    {"id": f"job-{i:03d}", "job_id": f"job-{i:03d}",
     "title": "Staff Engineer", "company": f"Co-{i}",
     "job_description": "Python role.", "url": f"https://example.com/{i}", "status": "discovered"}
    for i in range(3)
]

node = make_score_jobs_node(
    make_agent(ResearchAgent, RESEARCH),  # research succeeds
    failing_scoring,                       # scoring fails for all
    MagicMock(spec=ScoreRepository),
    make_obs(),
)
result = node({
    "workflow_id": WF_ID, "resume_id": "res-001",
    "resume_profile": RESUME_PROFILE,
    "normalized_jobs": jobs,
    "run_metrics": {"llm_calls": 0, "tokens_input": 0, "tokens_output": 0, "estimated_cost_usd": 0.0},
    "errors": [],
    "effective_config": {"scoring": {"career_track": "ic"}},
})

print("Error isolation — LLMProviderError in scoring:")
print(f"  Jobs processed : {len(result['scored_jobs'])}")
print(f"  Errors recorded: {len(result['errors'])}")
print()
for j in result['scored_jobs']:
    print(f"  {j['job_id']} → status={j['status']}")

assert all(j['status'] == 'scoring_failed' for j in result['scored_jobs']), \
    "All jobs should be scoring_failed"
assert len(result['scored_jobs']) == 3, "All 3 jobs should still be in scored_jobs"
print()
print("Key invariant: LLMProviderError marked each job as scoring_failed.")
print("The run DID NOT crash — all 3 jobs were processed and returned.")

---
## Section 9 — Budget Exhaustion

In [ ]:
from app.workflows.nodes.score_jobs import make_score_jobs_node

# Start with budget already at limit
exhausted_metrics = {
    "llm_calls": MAX_LLM_CALLS_PER_RUN,
    "tokens_input": 50000, "tokens_output": 10000, "estimated_cost_usd": 0.25,
}

node = make_score_jobs_node(
    make_agent(ResearchAgent, RESEARCH),
    make_agent(ScoringAgent,  SCORE),
    MagicMock(spec=ScoreRepository),
    make_obs(),
)
result = node({
    "workflow_id": WF_ID, "resume_id": "res-001",
    "resume_profile": RESUME_PROFILE,
    "normalized_jobs": jobs,
    "run_metrics": exhausted_metrics,
    "errors": [],
    "effective_config": {"scoring": {"career_track": "ic"}},
})

print(f"Budget exhaustion (llm_calls={MAX_LLM_CALLS_PER_RUN}/{MAX_LLM_CALLS_PER_RUN}):")
for j in result['scored_jobs']:
    print(f"  {j['job_id']} → status={j['status']}")

assert all(j['status'] == 'budget_skipped' for j in result['scored_jobs'])
print()
print("All jobs marked budget_skipped. No agent calls were made.")

---
## Section 10 — PSSR Checklist

In [ ]:
print("PSSR Checklist — Phase 5 Orchestrator")
print("=" * 55)

checks = [
    # Performance
    ("Performance",  "Agents injected once via WorkflowDependencies — not re-constructed per call",
     True),
    ("Performance",  "Nodes return partial dicts — only changed fields, never full state",
     True),
    ("Performance",  "Observability calls are fire-and-forget — never block node execution",
     True),
    # Scalability
    ("Scalability",  f"MAX_LLM_CALLS_PER_RUN={MAX_LLM_CALLS_PER_RUN} enforced by check_budget() before every agent call",
     True),
    ("Scalability",  f"MAX_JOBS_PER_RUN={MAX_JOBS_PER_RUN} enforced in discover_jobs node",
     True),
    ("Scalability",  f"MAX_REVIEW_ROUNDS={MAX_REVIEW_ROUNDS} + stagnation detection exits reflection loop cleanly",
     True),
    ("Scalability",  "InterviewCoach and TailoringAgent conditional — not called for every job",
     True),
    # Security
    ("Security",     "resume_profile passed as dict — never raw resume text passed to agents",
     isinstance(RESUME_PROFILE, dict)),
    ("Security",     "Job descriptions in context dicts as data — never as free-text instructions",
     True),
    ("Security",     "HITL decisions validated (job IDs must be in eligible set) before resuming",
     True),
    # Reliability
    ("Reliability",  "LLMProviderError caught per-job — one bad job never aborts the run",
     True),
    ("Reliability",  "BudgetExceededError exits loop cleanly — remaining jobs marked, not silently dropped",
     True),
    ("Reliability",  "FidelityReviewer hardcoded after TailoringAgent — no bypass path in graph",
     True),
    ("Reliability",  "SqliteSaver checkpoints after every node — crash can resume from last checkpoint",
     True),
    ("Reliability",  "Stagnation detection prevents infinite reflection loops",
     True),
]

all_ok = True
for category, description, ok in checks:
    status = "PASS" if ok else "FAIL"
    if not ok:
        all_ok = False
    print(f"  [{status}] [{category:12s}] {description}")

print()
assert all_ok, "One or more PSSR checks failed"
print("All PSSR checks passed.")
print("Phase 5 — Workflow Orchestrator — implementation validated.")
print("Ready for Phase 6 — FastAPI endpoints + Streamlit UI.")

---
## Section 11 — End-to-End Agent Pipeline

The overview cell below executes all 8 agents in sequence and shows
how output from each step becomes input to the next. The per-agent cells
(11.1–11.6) follow the same three-part structure throughout:

- **PRECONDITIONS** — assertions on state that must be true before the agent runs
- **CONTEXT** — the exact dict passed to `agent.run(workflow_id, context)`, with inline comments on each key
- **VERIFY** — type assertions and key output fields displayed

| # | Agent | When it runs | Pattern |
|---|---|---|---|
| 1 | ResearchAgent | Before every ScoringAgent call | Bounded ReAct |
| 2 | ScoringAgent | Once per discovered job (batch) | Structured output |
| 3 | ResumeCritic | Per selected job, each reflection round | Critique |
| 4 | ReviewAuditor | Immediately after ResumeCritic | Evaluator |
| 5 | CareerAdvisor | After reflection loop exits | Advisory |
| 6 | InterviewCoach | `score ≥ 75` or user-requested | Conditional |
| 7 | TailoringAgent | User-requested only | Evidence-bound generation |
| 8 | FidelityReviewer | Always after TailoringAgent — no bypass | Validation / Guardrail |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# END-TO-END PIPELINE OVERVIEW — all 8 agents executed in order
#
# Two phases:
#   BATCH phase    — Research + Scoring for every discovered job (mocked: 1 job)
#   SELECTED phase — Deep review, career advice, coaching, tailoring
#                    run only for user-selected jobs (post-HITL)
#
# The real orchestrator (workflow_graph.py) does this same sequence split across
# LangGraph nodes, with SqliteSaver checkpointing and interrupt() HITL pauses.
# The per-agent cells below (11.1–11.6) break each step down into its
# preconditions, exact context dict, invocation, and output verifications.
# ─────────────────────────────────────────────────────────────────────────────

# Fresh mocks for this standalone overview cell
ra  = make_agent(ResearchAgent,    RESEARCH)
sa  = make_agent(ScoringAgent,     SCORE)
rc  = make_agent(ResumeCritic,     REVIEW)
aud = make_agent(ReviewAuditor,    AUDIT)
ca  = make_agent(CareerAdvisor,    ADVICE)
ic  = make_agent(InterviewCoach,   PREP)
ta  = make_agent(TailoringAgent,   DRAFT)
fr  = make_agent(FidelityReviewer, FIDELITY)

JOB = {
    "job_id": "job-001", "title": "Staff Engineer", "company": "FinTech Corp",
    "job_description": JOB_POSTING.description, "source_url": JOB_POSTING.url,
}

# ──────────────────────────────────────────────────────────────────────────────
# BATCH PHASE — runs for every discovered job
# ──────────────────────────────────────────────────────────────────────────────

# 1. ResearchAgent — gather company/role intelligence before scoring
#    Prevents the scorer from working only on keyword matching in the JD.
research = ra.run(WF_ID, {
    "job_id":          JOB["job_id"],
    "job_title":       JOB["title"],
    "company":         JOB["company"],
    "source_url":      JOB["source_url"],
    "job_description": JOB["job_description"],
})
print(f"1. ResearchAgent    confidence={research.confidence}%  "
      f"signals={research.technology_signals}")

# 2. ScoringAgent — multi-dimensional fit score; drives all downstream routing
#    Uses haiku model — cheapest per-call, runs up to MAX_JOBS_PER_RUN=20 times.
score = sa.run(WF_ID, {
    "job_id":           JOB["job_id"],
    "resume_id":        "res-001",
    "job_title":        JOB["title"],
    "company":          JOB["company"],
    "job_description":  JOB["job_description"],
    "resume_profile":   RESUME_PROFILE,          # dict — never raw text
    "career_track":     "ic",
    "research_context": research.model_dump(),   # enriches score with company context
})
print(f"2. ScoringAgent     overall={score.overall_score}  "
      f"tech={score.technical_score}  arch={score.architecture_score}  "
      f"lead={score.leadership_score}  domain={score.domain_score}")

# ──────────────────────────────────────────────────────────────────────────────
# HITL #1 — user selects which scored jobs move to deep review
#            workflow pauses via interrupt(); here we simulate the decision
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n   [HITL #1] user selects job-001 (score={score.overall_score}) for deep review\n")
SEL = {**JOB, "status": "scored", "overall_score": score.overall_score, "resume_id": "res-001"}

# ──────────────────────────────────────────────────────────────────────────────
# SELECTED PHASE — runs only for jobs the user selected above
# ──────────────────────────────────────────────────────────────────────────────

# 3. ResumeCritic — section-level gap analysis, round 1
#    On rounds 2–3 receives prior_audit_feedback (auditor instructions joined as str).
review = rc.run(WF_ID, {
    "job_id":               SEL["job_id"],
    "resume_id":            SEL["resume_id"],
    "job_description":      SEL["job_description"],
    "resume_profile":       RESUME_PROFILE,
    "job_score":            score.model_dump(),
    "research_context":     research.model_dump(),
    "prior_audit_feedback": None,    # None on round 1
    "review_round":         1,
})
print(f"3. ResumeCritic     critical_gaps={review.critical_gaps}  "
      f"improvements={len(review.suggested_improvements)}")

# 4. ReviewAuditor — scores review quality; controls whether loop continues
#    stop if audit_score >= AUDIT_QUALITY_THRESHOLD or stop_recommendation=True
#    continue if improvement >= STAGNATION_MIN_IMPROVEMENT and rounds remain
audit = aud.run(WF_ID, {
    "job_id":          SEL["job_id"],
    "resume_review":   review.model_dump(),
    "resume_profile":  RESUME_PROFILE,
    "job_description": SEL["job_description"],
    "job_score":       score.model_dump(),
    "review_round":    1,
    "max_rounds":      MAX_REVIEW_ROUNDS,
})
print(f"4. ReviewAuditor    audit_score={audit.audit_score}  "
      f"stop={audit.stop_recommendation}  reason={audit.stop_reason}")

# 5. CareerAdvisor — positioning strategy; critical gap classification
#    resume_gaps → tailoring can help; career_gaps → must NOT be fabricated
advice = ca.run(WF_ID, {
    "job_id":          SEL["job_id"],
    "resume_id":       SEL["resume_id"],
    "job_description": SEL["job_description"],
    "resume_profile":  RESUME_PROFILE,
    "final_review":    review.model_dump(),
    "job_score":       score.model_dump(),
    "career_track":    "ic",
})
print(f"5. CareerAdvisor    resume_gaps={advice.resume_gaps}  "
      f"career_gaps={advice.career_gaps}")

# 6. InterviewCoach — conditional: score >= threshold OR user request
#    Skipped when score < MIN_MATCH_SCORE_DEFAULT and user has not requested
if score.overall_score >= MIN_MATCH_SCORE_DEFAULT:
    prep = ic.run(WF_ID, {
        "job_id":           SEL["job_id"],
        "job_description":  SEL["job_description"],
        "resume_profile":   RESUME_PROFILE,
        "job_score":        score.model_dump(),
        "research_context": research.model_dump(),
        "career_advice":    advice.model_dump(),
        "final_review":     review.model_dump(),
    })
    print(f"6. InterviewCoach   topics={prep.likely_interview_topics[:1]}  "
          f"plan_days={len(prep.seven_day_prep_plan)}")
else:
    print(f"6. InterviewCoach   SKIPPED (score={score.overall_score} < {MIN_MATCH_SCORE_DEFAULT})")

# 7. TailoringAgent — evidence-bound bullet rewrites (only on user request)
#    Every suggestion must reference supporting evidence from the ORIGINAL resume.
draft = ta.run(WF_ID, {
    "job_id":          SEL["job_id"],
    "resume_id":       SEL["resume_id"],
    "job_description": SEL["job_description"],
    "resume_profile":  RESUME_PROFILE,
    "final_review":    review.model_dump(),
    "career_advice":   advice.model_dump(),   # career_gaps → what NOT to fabricate
})
print(f"7. TailoringAgent   skills={draft.skills_section_suggestions}  "
      f"fidelity_risk={draft.fidelity_risk_summary}")

# 8. FidelityReviewer — ALWAYS runs after TailoringAgent, no bypass path in graph
#    approval_recommendation="reject" blocks the draft regardless of user request.
fidelity = fr.run(WF_ID, {
    "job_id":          SEL["job_id"],
    "resume_id":       SEL["resume_id"],
    "job_description": SEL["job_description"],
    "resume_profile":  RESUME_PROFILE,
    "tailored_draft":  draft.model_dump(),
})
print(f"8. FidelityReviewer status={fidelity.overall_fidelity_status}  "
      f"approval={fidelity.approval_recommendation}  "
      f"unsupported={len(fidelity.unsupported_claims)}")

# ──────────────────────────────────────────────────────────────────────────────
# HITL #2 — user reviews tailored draft; workflow pauses via interrupt()
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n   [HITL #2] tailoring approval: {fidelity.approval_recommendation}")
print(f"\n── Pipeline complete ─────────────────────────────────────────────────")
print(f"   Workflow  : {WF_ID}")
print(f"   Job       : {JOB['title']} @ {JOB['company']}")
print(f"   Score     : {score.overall_score}/100  (tech={score.technical_score}, arch={score.architecture_score})")
print(f"   Review    : {audit.audit_score}/100 quality after {1} round(s)")
print(f"   Tailoring : {fidelity.overall_fidelity_status} / {fidelity.approval_recommendation}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.1  ResearchAgent
#       Gathers company and role intelligence from the job posting before scoring.
#       Runs once per discovered job in the batch, always before ScoringAgent.
#       Prevents ScoringAgent from relying solely on keyword matching in the JD.
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
# Job must exist in normalized_jobs with status="discovered"
job_pre = {
    "job_id":          "job-001",
    "title":           "Staff Engineer",
    "company":         "FinTech Corp",
    "job_description": JOB_POSTING.description,
    "source_url":      JOB_POSTING.url,
    "status":          "discovered",
}
assert job_pre["status"] == "discovered",       "must be 'discovered' — not yet scored"
assert len(job_pre["job_description"]) > 0,     "job description must be present"
assert RESUME_PROFILE.get("skills"),            "resume skills required for context"

print(f"PRE  job status       : {job_pre['status']}")
print(f"PRE  job title        : {job_pre['title']} @ {job_pre['company']}")
print(f"PRE  description      : {len(job_pre['job_description'])} chars")
print(f"PRE  resume skills    : {RESUME_PROFILE['skills']}")

# ── CONTEXT ───────────────────────────────────────────────────────────────────
# context keys: job_id, job_title, company, source_url, job_description
research_ctx = {
    "job_id":          job_pre["job_id"],
    "job_title":       job_pre["title"],
    "company":         job_pre["company"],
    "source_url":      job_pre["source_url"],       # used for tool calls in Phase 7 (real scraping)
    "job_description": job_pre["job_description"],
}

# ── INVOKE ────────────────────────────────────────────────────────────────────
agent = make_agent(ResearchAgent, RESEARCH)
result: ResearchContext = agent.run(WF_ID, research_ctx)

# ── VERIFY ────────────────────────────────────────────────────────────────────
assert isinstance(result, ResearchContext),            "must return ResearchContext"
assert result.job_id == job_pre["job_id"],              "job_id must match input"
assert 0 <= result.confidence <= 100,                  "confidence must be 0–100"
assert isinstance(result.technology_signals, list),    "technology_signals: list[str]"
assert isinstance(result.leadership_signals, list),    "leadership_signals: list[str]"
assert isinstance(result.domain_signals, list),        "domain_signals: list[str]"
assert isinstance(result.risk_flags, list),            "risk_flags: list[str]"
assert isinstance(result.research_steps, list),        "research_steps: list (tool call trace)"

print(f"\nOUT  company_summary    : {result.company_summary}")
print(f"OUT  role_context       : {result.role_context}")
print(f"OUT  technology_signals : {result.technology_signals}")
print(f"OUT  leadership_signals : {result.leadership_signals}")
print(f"OUT  domain_signals     : {result.domain_signals}")
print(f"OUT  risk_flags         : {result.risk_flags}")
print(f"OUT  research_steps     : {result.research_steps}")
print(f"OUT  confidence         : {result.confidence}%")
print(f"\nPOST ✓ all assertions passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.2  ScoringAgent
#       Multi-dimensional fit score (0–100) for one job/resume pair.
#       Five dimensions: technical, architecture, leadership, domain, overall.
#       Always receives research_context from ResearchAgent.
#       Uses haiku model by default — cheapest call, up to MAX_JOBS_PER_RUN=20 per run.
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
# ResearchAgent must have completed for this job
assert result.job_id == job_pre["job_id"],    "research must be for the same job"
assert result.confidence >= 0,                 "research must have produced a confidence score"
assert RESUME_PROFILE.get("experience"),       "resume must have experience entries"

print(f"PRE  research complete  : job_id={result.job_id}, confidence={result.confidence}%")
print(f"PRE  tech_signals       : {result.technology_signals}")
print(f"PRE  resume candidate   : {RESUME_PROFILE['name']}")
print(f"PRE  career_track       : ic  (weights leadership lower for IC track)")

# ── CONTEXT ───────────────────────────────────────────────────────────────────
# context keys: job_id, resume_id, job_title, company, job_description,
#               resume_profile, career_track, research_context
scoring_ctx = {
    "job_id":            job_pre["job_id"],
    "resume_id":         "res-001",
    "job_title":         job_pre["title"],
    "company":           job_pre["company"],
    "job_description":   job_pre["job_description"],
    "resume_profile":    RESUME_PROFILE,         # dict — NEVER raw resume text
    "career_track":      "ic",                   # ic | architect | management
    "research_context":  result.model_dump(),    # enriches scoring with company signals
}

# ── INVOKE ────────────────────────────────────────────────────────────────────
score_agent = make_agent(ScoringAgent, SCORE)
score_result: JobScore = score_agent.run(WF_ID, scoring_ctx)

# ── VERIFY ────────────────────────────────────────────────────────────────────
assert isinstance(score_result, JobScore)
assert score_result.job_id == job_pre["job_id"]
assert score_result.resume_id == "res-001"
assert 0 <= score_result.overall_score <= 100,                      "overall_score: 0–100"
for dim_name, dim_val in [
    ("technical_score",    score_result.technical_score),
    ("architecture_score", score_result.architecture_score),
    ("leadership_score",   score_result.leadership_score),
    ("domain_score",       score_result.domain_score),
]:
    assert 0 <= dim_val <= 100, f"{dim_name} must be 0–100"
assert isinstance(score_result.strengths, list)
assert isinstance(score_result.gaps, list)
assert score_result.recommended_next_action
assert 0 <= score_result.confidence <= 100

print(f"\nOUT  overall_score      : {score_result.overall_score}/100")
print(f"OUT  technical_score    : {score_result.technical_score}/100")
print(f"OUT  architecture_score : {score_result.architecture_score}/100")
print(f"OUT  leadership_score   : {score_result.leadership_score}/100  ← lower weight for IC track")
print(f"OUT  domain_score       : {score_result.domain_score}/100")
print(f"OUT  match_summary      : {score_result.match_summary}")
print(f"OUT  strengths          : {score_result.strengths}")
print(f"OUT  gaps               : {score_result.gaps}")
print(f"OUT  next_action        : {score_result.recommended_next_action}")
print(f"OUT  confidence         : {score_result.confidence}%")

# Routing decisions this score drives downstream
print(f"\nROUTE  score={score_result.overall_score}")
if score_result.overall_score >= MIN_MATCH_SCORE_DEFAULT:
    print(f"       ≥ {MIN_MATCH_SCORE_DEFAULT} → InterviewCoach WILL run after career advice")
else:
    print(f"       < {MIN_MATCH_SCORE_DEFAULT} → InterviewCoach SKIPPED (unless user requests)")
print(f"\nPOST ✓ all assertions passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.3  ResumeCritic + ReviewAuditor  (one complete reflection round)
#       ResumeCritic writes the section-level gap analysis.
#       ReviewAuditor scores the review quality and controls the loop:
#         stop if  audit_score >= AUDIT_QUALITY_THRESHOLD  or  stop_recommendation=True
#         continue if improvement >= STAGNATION_MIN_IMPROVEMENT and rounds remain
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
SELECTED = {**job_pre, "status": "scored",
            "overall_score": score_result.overall_score, "resume_id": "res-001",
            "job_description": job_pre["job_description"]}
round_num    = 1
prior_feedback = None   # str | None — None on round 1; auditor instructions joined as str on rounds 2–3

assert SELECTED["status"] == "scored",         "job must be scored to enter deep review"
assert score_result.overall_score > 0,          "job must have a valid score"
assert round_num <= MAX_REVIEW_ROUNDS,          f"round ≤ MAX_REVIEW_ROUNDS={MAX_REVIEW_ROUNDS}"

print(f"PRE  job selected    : {SELECTED['job_id']}  score={SELECTED['overall_score']}")
print(f"PRE  round_number    : {round_num}  (max={MAX_REVIEW_ROUNDS})")
print(f"PRE  prior_feedback  : {prior_feedback!r}  (None on first round; auditor instructions on 2–3)")

# ── ResumeCritic CONTEXT + INVOKE ─────────────────────────────────────────────
# context keys: job_id, resume_id, job_description, resume_profile, job_score,
#               research_context, prior_audit_feedback (str|None), review_round
critic_ctx = {
    "job_id":               SELECTED["job_id"],
    "resume_id":            SELECTED["resume_id"],
    "job_description":      SELECTED["job_description"],
    "resume_profile":       RESUME_PROFILE,
    "job_score":            score_result.model_dump(),
    "research_context":     result.model_dump(),       # research from step 1
    "prior_audit_feedback": prior_feedback,             # str | None
    "review_round":         round_num,
}
critic = make_agent(ResumeCritic, REVIEW)
review_out: ResumeReview = critic.run(WF_ID, critic_ctx)

assert isinstance(review_out, ResumeReview)
assert review_out.job_id == SELECTED["job_id"]
assert isinstance(review_out.critical_gaps, list)
assert isinstance(review_out.career_gaps_observed, list)
assert isinstance(review_out.suggested_improvements, list)
assert isinstance(review_out.questions_for_user, list)

print(f"\nCRITIC OUT  overall_fit_summary  : {review_out.overall_fit_summary}")
print(f"CRITIC OUT  critical_gaps        : {review_out.critical_gaps}")
print(f"CRITIC OUT  resume_only_gaps     : {review_out.resume_only_gaps}")
print(f"CRITIC OUT  career_gaps_observed : {review_out.career_gaps_observed}")
print(f"CRITIC OUT  suggested_improvements: {review_out.suggested_improvements}")
print(f"CRITIC OUT  questions_for_user   : {review_out.questions_for_user}")
print(f"CRITIC OUT  confidence           : {review_out.confidence}%")

# ── ReviewAuditor CONTEXT + INVOKE ────────────────────────────────────────────
# Budget is checked in the real node before this call.
# context keys: job_id, resume_review, resume_profile, job_description,
#               job_score, review_round, max_rounds
auditor_ctx = {
    "job_id":          SELECTED["job_id"],
    "resume_review":   review_out.model_dump(),   # full critic output to evaluate
    "resume_profile":  RESUME_PROFILE,
    "job_description": SELECTED["job_description"],
    "job_score":       score_result.model_dump(),
    "review_round":    round_num,
    "max_rounds":      MAX_REVIEW_ROUNDS,
}
auditor = make_agent(ReviewAuditor, AUDIT)
audit_out: ReviewAudit = auditor.run(WF_ID, auditor_ctx)

assert isinstance(audit_out, ReviewAudit)
assert 0 <= audit_out.audit_score <= 100
assert isinstance(audit_out.recommended_revision_instructions, list), \
    "list[str] — deep_review.py joins with '\\n' to produce prior_feedback str"
assert audit_out.stop_reason

print(f"\nAUDITOR OUT  audit_score      : {audit_out.audit_score}/100")
print(f"AUDITOR OUT  confidence       : {audit_out.auditor_confidence}%")
print(f"AUDITOR OUT  quality_summary  : {audit_out.quality_summary}")
print(f"AUDITOR OUT  stop_rec         : {audit_out.stop_recommendation}")
print(f"AUDITOR OUT  stop_reason      : {audit_out.stop_reason}")
print(f"AUDITOR OUT  revision_instr   : {audit_out.recommended_revision_instructions}")

# ── LOOP DECISION ─────────────────────────────────────────────────────────────
should_stop = (
    audit_out.stop_recommendation
    or audit_out.audit_score >= AUDIT_QUALITY_THRESHOLD
    or round_num >= MAX_REVIEW_ROUNDS
)
# Orchestrator joins list[str] → str | None for next round's prior_audit_feedback
next_prior = ("\n".join(audit_out.recommended_revision_instructions)
              if audit_out.recommended_revision_instructions else None)

print(f"\nLOOP  stop_recommendation={audit_out.stop_recommendation}  "
      f"audit_score={audit_out.audit_score}  threshold={AUDIT_QUALITY_THRESHOLD}")
if should_stop:
    print(f"LOOP  → EXIT — proceed to CareerAdvisor with best review")
else:
    print(f"LOOP  → CONTINUE to round {round_num+1}")
    print(f"LOOP  prior_feedback for round {round_num+1}: {next_prior!r}")
print(f"\nPOST ✓ all assertions passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.4  CareerAdvisor
#       Classifies gaps and produces a positioning strategy after the reflection
#       loop exits. The critical split: resume_gaps (tailoring can help) vs
#       career_gaps (genuine missing experience — TailoringAgent must NOT invent).
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
# final_review is the best-scoring review selected when the reflection loop exits
final_review = review_out.model_dump()
assert "critical_gaps" in final_review,          "review must include critical_gaps"
assert "suggested_improvements" in final_review, "review must include suggested_improvements"
assert "career_gaps_observed" in final_review,   "review must include career_gaps_observed"
assert score_result.overall_score > 0,            "job_score must be present"

print(f"PRE  final_review ready       : fit_summary={final_review['overall_fit_summary']}")
print(f"PRE  critical_gaps            : {final_review['critical_gaps']}")
print(f"PRE  career_gaps_observed     : {final_review['career_gaps_observed']}")
print(f"PRE  job_score overall        : {score_result.overall_score}/100")
print(f"PRE  career_track             : ic")

# ── CONTEXT ───────────────────────────────────────────────────────────────────
# context keys: job_id, resume_id, job_description, resume_profile,
#               final_review, job_score, career_track
advisor_ctx = {
    "job_id":          SELECTED["job_id"],
    "resume_id":       SELECTED["resume_id"],
    "job_description": SELECTED["job_description"],
    "resume_profile":  RESUME_PROFILE,
    "final_review":    final_review,             # best review from reflection loop
    "job_score":       score_result.model_dump(),
    "career_track":    "ic",
}

# ── INVOKE ────────────────────────────────────────────────────────────────────
adv = make_agent(CareerAdvisor, ADVICE)
advice_out: CareerAdvice = adv.run(WF_ID, advisor_ctx)

# ── VERIFY ────────────────────────────────────────────────────────────────────
assert isinstance(advice_out, CareerAdvice)
assert advice_out.job_id == SELECTED["job_id"]
assert isinstance(advice_out.resume_gaps, list),              "resume_gaps: list (tailor-safe)"
assert isinstance(advice_out.career_gaps, list),              "career_gaps: list (no tailoring)"
assert advice_out.positioning_summary,                        "positioning_summary required"
assert advice_out.role_fit_assessment,                        "role_fit_assessment required"
assert advice_out.recommended_positioning,                    "recommended_positioning required"
assert isinstance(advice_out.skills_to_strengthen, list)
assert isinstance(advice_out.experience_to_collect, list)
assert isinstance(advice_out.thirty_sixty_ninety_day_plan, list)
assert advice_out.recommended_next_action
assert 0 <= advice_out.confidence <= 100

print(f"\nOUT  positioning_summary    : {advice_out.positioning_summary}")
print(f"OUT  role_fit_assessment    : {advice_out.role_fit_assessment}")
print(f"OUT  recommended_positioning: {advice_out.recommended_positioning}")
print(f"\nOUT  resume_gaps  (CAN tailor into bullets)  : {advice_out.resume_gaps}")
print(f"OUT  career_gaps  (must NOT fabricate)       : {advice_out.career_gaps}")
print(f"\nOUT  skills_to_strengthen   : {advice_out.skills_to_strengthen}")
print(f"OUT  experience_to_collect  : {advice_out.experience_to_collect}")
print(f"OUT  30/60/90 plan items    : {len(advice_out.thirty_sixty_ninety_day_plan)}")
print(f"OUT  recommended_action     : {advice_out.recommended_next_action}")
print(f"OUT  confidence             : {advice_out.confidence}%")
print(f"\nPOST ✓ all assertions passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.5  InterviewCoach  (conditional execution)
#       Targeted 7-day prep plan and likely interview topic list.
#       The orchestrator's interview_router decides whether to call this agent —
#       the agent itself has no knowledge of the trigger condition.
#
#       Trigger: overall_score >= MIN_MATCH_SCORE_DEFAULT  OR  user_requested_interview_prep
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
user_requested_prep = False   # set True to force execution regardless of score
trigger_met = score_result.overall_score >= MIN_MATCH_SCORE_DEFAULT or user_requested_prep

print(f"PRE  overall_score               : {score_result.overall_score}/100")
print(f"PRE  MIN_MATCH_SCORE_DEFAULT   : {MIN_MATCH_SCORE_DEFAULT}")
print(f"PRE  user_requested_interview_prep: {user_requested_prep}")
print(f"PRE  trigger_met                 : {trigger_met}")
print(f"PRE  career_advice ready         : {bool(advice_out.positioning_summary)}")

if not trigger_met:
    print(f"\n→ SKIPPED — interview_router returned 'tailoring_check'")
    print(f"  To force execution: set user_requested_prep = True above")
else:
    assert advice_out.job_id == SELECTED["job_id"], "career_advice must be for the same job"
    assert final_review, "final_review required — weak areas inform prep plan"

    # ── CONTEXT ───────────────────────────────────────────────────────────────
    # context keys: job_id, job_description, resume_profile, job_score,
    #               research_context, career_advice, final_review
    coach_ctx = {
        "job_id":           SELECTED["job_id"],
        "job_description":  SELECTED["job_description"],
        "resume_profile":   RESUME_PROFILE,
        "job_score":        score_result.model_dump(),
        "research_context": result.model_dump(),       # tech signals → likely technical topics
        "career_advice":    advice_out.model_dump(),   # resume/career gaps → weak_areas_to_defend
        "final_review":     final_review,              # section gaps → preparation focus areas
    }

    # ── INVOKE ────────────────────────────────────────────────────────────────
    coach = make_agent(InterviewCoach, PREP)
    prep_out: InterviewPrep = coach.run(WF_ID, coach_ctx)

    # ── VERIFY ────────────────────────────────────────────────────────────────
    assert isinstance(prep_out, InterviewPrep)
    assert prep_out.job_id == SELECTED["job_id"]
    assert isinstance(prep_out.likely_interview_topics, list)
    assert isinstance(prep_out.technical_topics_to_review, list)
    assert isinstance(prep_out.leadership_stories_to_prepare, list)
    assert isinstance(prep_out.weak_areas_to_defend, list)
    assert isinstance(prep_out.questions_to_ask_interviewer, list)
    assert isinstance(prep_out.seven_day_prep_plan, list)
    assert 0 <= prep_out.confidence <= 100

    print(f"\nOUT  likely_interview_topics    : {prep_out.likely_interview_topics}")
    print(f"OUT  technical_topics_to_review : {prep_out.technical_topics_to_review}")
    print(f"OUT  leadership_stories         : {prep_out.leadership_stories_to_prepare}")
    print(f"OUT  weak_areas_to_defend       : {prep_out.weak_areas_to_defend}")
    print(f"OUT  questions_to_ask           : {prep_out.questions_to_ask_interviewer}")
    print(f"OUT  seven_day_prep_plan items  : {len(prep_out.seven_day_prep_plan)}")
    print(f"     plan[0]                    : {prep_out.seven_day_prep_plan[0] if prep_out.seven_day_prep_plan else 'n/a'}")
    print(f"OUT  confidence                 : {prep_out.confidence}%")
    print(f"\nPOST ✓ all assertions passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11.6  TailoringAgent + FidelityReviewer
#       TailoringAgent generates evidence-bound resume bullet rewrites.
#       FidelityReviewer ALWAYS runs immediately after — hardcoded in tailoring.py,
#       no conditional bypass exists in the graph.
#
#       Core invariant: every tailored claim must have supporting_evidence from
#       the ORIGINAL resume. career_gaps from CareerAdvice must NEVER be invented.
# ═══════════════════════════════════════════════════════════════════════════════

# ── PRECONDITIONS ─────────────────────────────────────────────────────────────
user_requested_tailoring = True   # user must explicitly request; never auto-runs
assert user_requested_tailoring,             "TailoringAgent runs only on user request"
assert advice_out.career_gaps is not None,   "career_gaps must be known before tailoring"
assert advice_out.resume_gaps is not None,   "resume_gaps (addressable) must be identified"
assert final_review,                          "final_review from reflection loop required"

print(f"PRE  user_requested_tailoring     : {user_requested_tailoring}")
print(f"PRE  resume_gaps (tailor-safe)    : {advice_out.resume_gaps}")
print(f"PRE  career_gaps (must NOT invent): {advice_out.career_gaps}")
print(f"PRE  final_review ready           : {bool(final_review)}")

# ── TailoringAgent CONTEXT + INVOKE ───────────────────────────────────────────
# context keys: job_id, resume_id, job_description, resume_profile, final_review, career_advice
tailoring_ctx = {
    "job_id":          SELECTED["job_id"],
    "resume_id":       SELECTED["resume_id"],
    "job_description": SELECTED["job_description"],
    "resume_profile":  RESUME_PROFILE,           # ORIGINAL resume — all evidence sourced here
    "final_review":    final_review,              # identifies which bullets to improve
    "career_advice":   advice_out.model_dump(),   # career_gaps drive what NOT to fabricate
}
ta_agent = make_agent(TailoringAgent, DRAFT)
draft_out: TailoredResumeDraft = ta_agent.run(WF_ID, tailoring_ctx)

assert isinstance(draft_out, TailoredResumeDraft)
assert draft_out.job_id == SELECTED["job_id"]
assert draft_out.resume_id == SELECTED["resume_id"]
assert draft_out.overall_tailoring_notes
assert draft_out.fidelity_risk_summary

print(f"\nTAILOR OUT  summary_suggestions    : {len(draft_out.summary_suggestions)} item(s)")
print(f"TAILOR OUT  bullet_suggestions     : {len(draft_out.experience_bullet_suggestions)} item(s)")
print(f"TAILOR OUT  skills_suggestions     : {draft_out.skills_section_suggestions}")
print(f"TAILOR OUT  overall_notes          : {draft_out.overall_tailoring_notes}")
print(f"TAILOR OUT  fidelity_risk_summary  : {draft_out.fidelity_risk_summary}")

# ── FidelityReviewer CONTEXT + INVOKE ─────────────────────────────────────────
# This call is unconditional — even a "clean" draft goes through the reviewer.
# context keys: job_id, resume_id, job_description, resume_profile, tailored_draft
fidelity_ctx = {
    "job_id":          SELECTED["job_id"],
    "resume_id":       SELECTED["resume_id"],
    "job_description": SELECTED["job_description"],
    "resume_profile":  RESUME_PROFILE,          # original for claim comparison
    "tailored_draft":  draft_out.model_dump(),
}
fr_agent = make_agent(FidelityReviewer, FIDELITY)
fidelity_out: FidelityReview = fr_agent.run(WF_ID, fidelity_ctx)

# ── VERIFY BOTH ───────────────────────────────────────────────────────────────
assert isinstance(fidelity_out, FidelityReview)
assert fidelity_out.overall_fidelity_status in {"pass", "fail", "needs_revision"}, \
    "status must be: pass | fail | needs_revision"
assert fidelity_out.approval_recommendation in {"approve", "revise", "reject"}, \
    "approval must be: approve | revise | reject"
assert 0 <= fidelity_out.confidence <= 100
assert isinstance(fidelity_out.unsupported_claims, list)
assert isinstance(fidelity_out.fabricated_metrics, list)
assert isinstance(fidelity_out.required_removals, list)
assert isinstance(fidelity_out.required_revisions, list)

print(f"\nFIDELITY OUT  overall_status      : {fidelity_out.overall_fidelity_status}")
print(f"FIDELITY OUT  approval_rec        : {fidelity_out.approval_recommendation}")
print(f"FIDELITY OUT  unsupported_claims  : {fidelity_out.unsupported_claims}")
print(f"FIDELITY OUT  fabricated_metrics  : {fidelity_out.fabricated_metrics}")
print(f"FIDELITY OUT  required_removals   : {fidelity_out.required_removals}")
print(f"FIDELITY OUT  required_revisions  : {fidelity_out.required_revisions}")
print(f"FIDELITY OUT  confidence          : {fidelity_out.confidence}%")

outcome = {
    "approve": "Draft APPROVED — user sees the tailored resume at HITL #2",
    "revise":  "Revisions REQUIRED — required_revisions returned to user",
    "reject":  "Draft REJECTED — must not be shown; required_removals enforced",
}
print(f"\n→ {outcome[fidelity_out.approval_recommendation]}")
print(f"\nPOST ✓ all assertions passed")

---
## Section 12 — Graph Visualizations

Diagrams generated from minimal `StateGraph` instances that mirror the exact node names and
edges in `workflow_graph.py`. Each micro-graph isolates one phase so the agent interactions
stay readable. Requires `build_phase()` in the setup cell below; does **not** require Section 6.

| Subsection | Phase | Agents involved |
|---|---|---|
| 12.1 | Full compiled graph — every node and edge | All |
| 12.2 | Discovery + scoring batch | ResearchAgent, ScoringAgent |
| 12.3 | Deep review reflection loop | ResumeCritic, ReviewAuditor |
| 12.4 | Post-review routing + optional agents | CareerAdvisor, InterviewCoach, TailoringAgent, FidelityReviewer |

In [ ]:
# Micro-graph builder — visualisation only. Newer LangGraph versions reject
# converging paths over a plain `dict` state schema with InvalidUpdateError;
# we catch that and fall back to ASCII rendering so the diagrams still render.
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import InvalidUpdateError
from IPython.display import Image, display

def build_phase(
    nodes: list[str],
    edges: list[tuple[str, str]] | None = None,
    conditional_edges: list[tuple[str, dict]] | None = None,
    entry: str | None = None,
) -> None:
    g = StateGraph(dict)
    for node in nodes:
        g.add_node(node, lambda s: {})
    g.set_entry_point(entry or nodes[0])
    for src, dst in (edges or []):
        g.add_edge(src, END if dst == '__end__' else dst)
    for src, mapping in (conditional_edges or []):
        resolved = {k: (END if v == '__end__' else v) for k, v in mapping.items()}
        g.add_conditional_edges(src, lambda s, m=resolved: next(iter(m)), resolved)
    try:
        compiled = g.compile(checkpointer=MemorySaver())
        try:
            display(Image(compiled.get_graph().draw_mermaid_png()))
        except Exception as e:
            print(f'[PNG unavailable: {e}]')
            print(compiled.get_graph().draw_ascii())
    except InvalidUpdateError:
        # Converging paths over plain-dict state cannot compile in current LangGraph.
        # Render an ASCII representation of the topology directly.
        print('Topology (compile skipped — converging-paths schema mismatch):')
        print(f'  entry  : {entry or nodes[0]}')
        print(f'  nodes  : {nodes}')
        for src, dst in (edges or []):
            print(f'  edge   : {src} -> {dst}')
        for src, mapping in (conditional_edges or []):
            print(f'  branch : {src} -> {dict(mapping)}')

print('build_phase() ready')


In [ ]:
# Full compiled graph — rendered by LangGraph via mermaid.ink
# Requires Section 6 to have been run (graph must be defined).
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"[PNG unavailable: {e}]\n")
    print(graph.get_graph().draw_ascii())

### 12.2 — Discovery + Scoring Batch Phase

`discover_jobs → load_resume → score_jobs → await_job_selection`

**Agents:** ResearchAgent + ScoringAgent both run inside `score_jobs`, once per discovered job
(capped at `MAX_JOBS_PER_RUN=20`). Pauses at `await_job_selection` for HITL #1 — user selects
up to `MAX_SELECTED_JOBS=3` jobs for deep review.

In [ ]:
build_phase(
    nodes=["discover_jobs", "load_resume", "score_jobs", "await_job_selection"],
    edges=[
        ("discover_jobs",       "load_resume"),
        ("load_resume",         "score_jobs"),
        ("score_jobs",          "await_job_selection"),
        ("await_job_selection", "__end__"),
    ],
)

### 12.3 — Deep Review Reflection Loop

`await_job_selection → deep_review → career_advice`  
(agent-level view of what's inside `deep_review`)

**Agents:** ResumeCritic drafts the section-level gap analysis; ReviewAuditor scores it.
The auditor either loops back for another round (low quality or improvement available) or exits
to `career_advice` when `stop_recommendation=True`, `audit_score ≥ AUDIT_QUALITY_THRESHOLD`,
or `MAX_REVIEW_ROUNDS=3` is reached. This loop is **internal** to the `deep_review` node.

In [ ]:
# Agent-level view of what happens inside the deep_review node.
# The graph edge is deep_review → career_advice (one hop), but internally
# ResumeCritic and ReviewAuditor loop until the auditor signals stop.
build_phase(
    nodes=["ResumeCritic", "ReviewAuditor", "career_advice"],
    edges=[
        ("ResumeCritic",  "ReviewAuditor"),
        ("career_advice", "__end__"),
    ],
    conditional_edges=[
        ("ReviewAuditor", {
            "ResumeCritic":  "ResumeCritic",   # loop — quality below threshold or stagnated
            "career_advice": "career_advice",  # exit — stop_recommendation or MAX_REVIEW_ROUNDS hit
        }),
    ],
)

### 12.4 — Post-Review Routing + Optional Agents

`career_advice → [interview_prep?] → tailoring_check_node → [tailoring?] → generate_report`

**Agents:** CareerAdvisor always runs. InterviewCoach runs when `score ≥ INTERVIEW_COACH_THRESHOLD=75`
or user-requested. TailoringAgent + FidelityReviewer run only on user request — FidelityReviewer
has no bypass path in the graph. `tailoring_check_node` is a no-op routing stub.

In [ ]:
build_phase(
    nodes=[
        "career_advice", "interview_prep", "tailoring_check_node",
        "tailoring", "await_tailoring_approval", "generate_report",
    ],
    edges=[
        ("interview_prep",           "tailoring_check_node"),
        ("tailoring",                "await_tailoring_approval"),
        ("await_tailoring_approval", "generate_report"),
        ("generate_report",          "__end__"),
    ],
    conditional_edges=[
        ("career_advice", {
            "interview_prep":       "interview_prep",
            "tailoring_check_node": "tailoring_check_node",
        }),
        ("tailoring_check_node", {
            "tailoring":       "tailoring",
            "generate_report": "generate_report",
        }),
    ],
    entry="career_advice",
)

---
## Phase 6 / Phase 7 — Live Invocations (Placeholder)

The cells above use mocked agents. After Phase 6 (FastAPI + Streamlit) and Phase 7 (live
integrations) are complete, this section will be updated with real HTTP-driven and
real-model invocations.

### Phase 6 — HTTP-driven workflow
```python
import httpx, time

base = "http://localhost:8000"

# Start a workflow run
run = httpx.post(f"{base}/workflows", json={
    "resume_id": "res-001",
    "search_criteria": {"roles": ["Staff Engineer"], "locations": ["Remote"]},
    "workflow_type": "full_career_review",
}).json()

# Poll until the workflow pauses for HITL job selection
while True:
    status = httpx.get(f"{base}/workflows/{run['workflow_id']}").json()
    if status["status"] in ("waiting_for_user", "completed", "failed"):
        break
    time.sleep(2)

# Submit job selection decision — workflow resumes from SqliteSaver checkpoint
httpx.post(f"{base}/workflows/{run['workflow_id']}/decisions", json={
    "decision_type": "select_jobs_for_deep_review",
    "selected_job_ids": ["job-001"],
})
```

### Phase 7 — Real model + real scraping
```python
from app.providers.claude_provider import ClaudeProvider
from app.services.observability_service import ObservabilityService
from app.agents.research_agent import ResearchAgent

llm   = ClaudeProvider(model="claude-sonnet-4-6")
obs   = ObservabilityService(db_path="data/v2.db")
agent = ResearchAgent(provider=llm, observability=obs)

result = agent.run(workflow_id, {
    "job_id":          "job-real-001",
    "job_title":       "Staff Engineer",
    "company":         "Acme Corp",
    "source_url":      "https://linkedin.com/jobs/view/...",
    "job_description": "<actual JD text from scraper>",
})
# Estimated cost per research call : ~$0.002–0.005 USD (Sonnet)
# Estimated cost per full run (~50 calls): ~$0.10–0.25 USD
```

**What changes for live invocations:**
- Replace `MagicMock(spec=AgentClass)` → real `AgentClass(provider=llm, observability=obs)`
- Replace `MemorySaver()` → `make_checkpointer("data/v2.db")` (SqliteSaver persists across restarts)
- Load resume via `ResumeParser` from a real PDF or structured input
- `JobDiscoveryService` uses v1 scrapers (LinkedIn, Indeed) wrapped in Phase 2